In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "earthaccess-auth>=0.4.0",
#     "icechunk>=2.2.2",
#     "matplotlib>=3.11.2",
#     "xarray>=2026.7.0",
# ]
# ///

# Quickstart: reading a store from anywhere

The [first quickstart](../quickstart/) reads chunks from ASDC's S3 bucket,
which works only in AWS us-west-2. This one reads the same chunks over
HTTPS from NASA's distribution endpoint, so it runs on a laptop.

It relies on the store's references being relative, as
`vcc://asdc/TEMPO/...`. Icechunk resolves such a reference through
whichever container is named `asdc`. This notebook replaces that container
with one that points at `https://data.asdc.earthdata.nasa.gov/` and sends
an Earthdata Login bearer token with each request. The HTTPS endpoint
serves the same keys as the bucket. It sits behind CloudFront, which
rate-limits it, so expect it to be slower than reading in-region.

Run it with [juv](https://github.com/manzt/juv):
`juv run docs/examples/quickstart-https.ipynb`.

## Authenticate

`login()` looks for `EARTHDATA_TOKEN` (or `EARTHDATA_USERNAME` and
`EARTHDATA_PASSWORD`), then `~/.netrc`, then prompts. Only the bearer token
is used here, so the base package is enough.

In [ ]:
import earthaccess_auth

token = earthaccess_auth.login().token["access_token"]

## Open the store

Icechunk opens the repository on Source Cooperative anonymously. The
repository's own config points the `asdc` container at S3. Swapping in an
HTTPS container with the same name sends every relative reference to
the HTTPS endpoint instead. `HttpAccess` authorizes the container to be
read. The token travels in the container's headers.

The token is a static header. Open the store again when it expires.

In [ ]:
import icechunk
import xarray as xr

storage = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix="pangeo/tempo-virtual-icechunk/tempo/no2/v04",
    region="us-west-2",
    anonymous=True,
)
repo = icechunk.Repository.open(storage)

https_prefix = "https://data.asdc.earthdata.nasa.gov/asdc-prod-protected/"
config = repo.config
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        https_prefix,
        icechunk.http_store(headers={"Authorization": f"Bearer {token}"}),
        name="asdc",
    )
)
repo = repo.reopen(
    config=config,
    authorize_virtual_chunk_access=icechunk.containers_credentials(
        {https_prefix: icechunk.credentials.HttpAccess}
    ),
)
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks=None)
ds

## Load a single point

Selecting a point and a time, then asking for the value, reads the one
chunk that holds it, over HTTPS. `main_data_quality_flag` is 0 for normal
retrievals.

In [ ]:
nyc = dict(latitude=40.7, longitude=-74.0, method="nearest")
scan = ds.sel(time="2026-09-03T18:00", method="nearest")

no2 = scan.vertical_column_troposphere.sel(**nyc)
flag = scan.main_data_quality_flag.sel(**nyc)
when = scan.time.values.astype("datetime64[m]")
print(f"{when}: {float(no2):.3g} {no2.units}, quality flag {int(flag)}")